# ==========================
# VERSION 1.0 
# Date: 2026-05-20
# ==========================

In [1]:
# Python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Ignore Unimportant Ignores
import warnings
warnings.filterwarnings('ignore')

In [2]:
train_path = '/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps/train_FD001.txt'
test_path = '/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps/test_FD001.txt'
rul_path = '/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps/RUL_FD001.txt'

In [3]:
df_train = pd.read_csv(train_path, sep= '\s+', header= None)
df_test = pd.read_csv(test_path, sep= '\s+', header=None)
df_rul = pd.read_csv(rul_path, sep='\s+', header=None, names=['RUL'])

print(f'Train data format: {df_train.shape}')
print(f'Test data format: {df_test.shape}')
print(f'RUL data format: {df_rul.shape}')

Train data format: (20631, 26)
Test data format: (13096, 26)
RUL data format: (100, 1)


In [4]:
column_names = ['unit_number', 'time_cycles', 'op_setting_1',
                'op_setting_2', 'op_setting_3']

for i in range(1, 22):
    column_names.append(f'sensor_{i}')

df_train.columns = column_names
df_test.columns = column_names

print("\nColumn names:")
print(df_train.columns.tolist())
print(f"\nNumber of columns: {len(df_train.columns)}")


Column names:
['unit_number', 'time_cycles', 'op_setting_1', 'op_setting_2', 'op_setting_3', 'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21']

Number of columns: 26


In [5]:
# REMOVE CONSTANT COLUMNS
constant_cols = []
for col in df_train.columns:
    if df_train[col].nunique() == 1 and col not in ['unit_number', 'time_cycles']:
        constant_cols.append(col)

print(f"Removed constant columns: {constant_cols}")
df_train = df_train.drop(columns=constant_cols)
df_test = df_test.drop(columns=constant_cols)

Removed constant columns: ['op_setting_3', 'sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']


In [6]:
# Quick statistics
print("\nTraining data statistics:")
print(df_train[['unit_number', 'time_cycles']].describe())

# How many engines/motors in training?
print(f"\nNumber of motors in training: {df_train['unit_number'].nunique()}")
print(f"Number of motors in testing: {df_test['unit_number'].nunique()}")


Training data statistics:
        unit_number   time_cycles
count  20631.000000  20631.000000
mean      51.506568    108.807862
std       29.227633     68.880990
min        1.000000      1.000000
25%       26.000000     52.000000
50%       52.000000    104.000000
75%       77.000000    156.000000
max      100.000000    362.000000

Number of motors in training: 100
Number of motors in testing: 100


In [7]:
# Calculate RUL for training
max_cycles = df_train.groupby('unit_number')['time_cycles'].max().reset_index()
max_cycles.columns = ['unit_number', 'max_cycle']

df_train = df_train.merge(max_cycles, on='unit_number', how='left')
df_train['RUL'] = df_train['max_cycle'] - df_train['time_cycles']

print(df_train[['unit_number', 'time_cycles', 'max_cycle', 'RUL']].head(10))

   unit_number  time_cycles  max_cycle  RUL
0            1            1        192  191
1            1            2        192  190
2            1            3        192  189
3            1            4        192  188
4            1            5        192  187
5            1            6        192  186
6            1            7        192  185
7            1            8        192  184
8            1            9        192  183
9            1           10        192  182


In [8]:
# Feature selection
feature_cols = [col for col in df_train.columns 
                if col not in ['unit_number', 'time_cycles', 'max_cycle', 'RUL']]

X_train = df_train[feature_cols].values
y_train = df_train['RUL'].values
X_test = df_test[feature_cols].values

In [9]:
df_test_last_cycle = df_test.groupby('unit_number').last().reset_index()
print(f"\nTest data after taking the last cycle for each engine:")
print(f"  Shape: {df_test_last_cycle.shape}")
print(f"  Number of engines: {df_test_last_cycle['unit_number'].nunique()}")

# Prepare X_test (same features)
X_test = df_test_last_cycle[feature_cols].values

print(f"\nX_test shape: {X_test.shape}")  # Should be (100, num_features)


Test data after taking the last cycle for each engine:
  Shape: (100, 19)
  Number of engines: 100

X_test shape: (100, 17)


In [10]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [11]:
model = RandomForestRegressor(n_estimators=400, max_depth=20, random_state=42)
model.fit(X_train_scaled, y_train)

RandomForestRegressor(max_depth=20, n_estimators=400, random_state=42)

In [12]:
predictions = model.predict(X_test_scaled)
predictions = np.maximum(predictions, 0)
predictions = np.round(predictions).astype(int)

In [13]:
y_true = df_rul['RUL'].values
rmse = np.sqrt(mean_squared_error(y_true, predictions))
mae = mean_absolute_error(y_true, predictions)
print(f"RMSE: {rmse:.2f}")

RMSE: 33.60
